# **CS 425 Final Project**
The purpose of this project is to visualize and measure the differences in accuracy and effectiveness between various machine learning architectures and methods. 

For section 1 of the project, we will look at how a standard Convolutional Neural Network (CNN) handles the prediction of read coverage in DNA accessibility versus a transformer architecture such as PDLLM or DNABERT-2.

For section 2 of the project, we will evaluate the performance differences in predicting enhancer activity in Arabidopsis between a standard CNN, a CNN with transfer learning, and a fine-tuned transformer.

The link to our project proposal, timelines, and detailed layout our plans is [here](https://docs.google.com/document/d/1emR3MxvZgMj_qGZ2zw8oQoUpwPpePoVyOWwxj0TpfPU/edit?tab=t.0). Additionally, the download for the dataset we're working with is available on [zenodo](https://zenodo.org/records/19509176) , which will need to be downloaded manually to run the cells, as the folder is too big to upload to Github without LFS.

In [5]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np

## DNA Accessibility ##

#### Data Pre-Processing ####
Done by Dev Bourland

In [ ]:
import pyBigWig
import os
from multiprocessing import Pool

winsize = 2000
windows = []

# original code + modifications from claude to support multithreading
for file in os.scandir("data/new_comb_data"):
    if file.name.endswith(".bw"):
        bw = pyBigWig.open(file.path)
        for chr in bw.chroms():
            for start in range(0, bw.chroms(chr) - winsize + 1, winsize):
                windows.append((chr, start, start + winsize))
        bw.close()
        break

def process_file(filepath):
    print(f"Processing {os.path.basename(filepath)} ...")
    bw = pyBigWig.open(filepath)
    exp = []
    for chrom, start, end in windows:
        center       = (start + end) // 2
        center_start = center - 100
        center_end   = center + 100
        bins = bw.stats(chrom, center_start, center_end, type="mean", nBins=10)
        exp.append([b if b is not None else 0.0 for b in bins])
    bw.close()
    return exp

filepaths = [f.path for f in os.scandir("data/new_comb_data") if f.name.endswith(".bw")]

print(f"Processing {len(filepaths)} files across {os.cpu_count()} cores ...")

with Pool(processes=os.cpu_count()) as pool:
    data = pool.map(process_file, filepaths)

Y = np.stack(data, axis=1)
Y = np.log1p(Y)

print(Y.shape)

np.savez_compressed("dataset.npz", Y=Y, windows=windows)

Processing 64 files across 12 cores ...
Processing SRX1096549_Rep0.rpgc.bw ...Processing SRX2240741.bw ...Processing SRX215438.bw ...Processing SRX1012884.bw ...Processing SRX1098136_Rep0.rpgc.bw ...Processing SRX1412761.bw ...Processing SRX1098138_Rep0.rpgc.bw ...Processing SRX2140205.bw ...Processing SRX130305.bw ...Processing SRX2467594.bw ...Processing DRX092570.bw ...Processing SRX1096551_Rep0.rpgc.bw ...











Processing SRX1096548_Rep0.rpgc.bw ...
Processing SRX3152333.bw ...
Processing SRX2311142.bw ...
Processing SRX1098137_Rep0.rpgc.bw ...
Processing SRX1098135_Rep0.rpgc.bw ...
Processing SRX215433.bw ...
Processing SRX1161361.bw ...
Processing SRX2140200.bw ...
Processing SRX1096550_Rep0.rpgc.bw ...
Processing SRX215474.bw ...
Processing SRX1012869.bw ...
Processing SRX130311.bw ...
Processing SRX3753856.bw ...
Processing SRX391990_Rep0.rpgc.bw ...
Processing SRX391992_Rep0.rpgc.bw ...
Processing SRX391994_Rep0.rpgc.bw ...
Processing SRX391996_Rep0.rpgc.bw ...
Processin

In [7]:
"""
A parser for FASTA files.

It can handle files that are local or on the web.
Gzipped files do not need to be unzipped.
"""

# USING FROM 02_MOTIF_PREP ASSIGNMENT.

import os
from urllib.request import urlopen

def myopen(fileName) :

    if not ( os.path.exists(fileName) and os.path.isfile(fileName) ):
        raise ValueError('file does not exist at %s' % fileName)
    
    import gzip
    fileHandle = gzip.GzipFile(fileName)

    gzippedFile = True
    try :
        line = fileHandle.readline()
        fileHandle.close()
    except :
        gzippedFile = False

    if gzippedFile :
        return gzip.GzipFile(fileName)
    else :
        return open(fileName)


class MalformedInput :
    "Exception raised when the input file does not look like a fasta file."
    pass

class FastaRecord :
    """Represents a record in a fasta file."""
    def __init__(self, header, sequence):
        """Create a record with the given header and sequence."""
        self.header = header
        self.sequence = sequence
    def __str__(self) :
        return '>' + self.header + '\n' + self.sequence + '\n'

    
def _fasta_itr_from_file(file_handle) :
    "Provide an iteration through the fasta records in file."

    h = file_handle.readline()[:-1]
    if h[0] != '>':
        raise MalformedInput()
    h = h[1:]

    seq = []
    for line in file_handle:
        line = line[:-1] # remove newline
        if line[0] == '>':
            yield FastaRecord(h,''.join(seq))
            h = line[1:]
            seq = []
            continue
        seq.append(line)

    yield FastaRecord(h,''.join(seq))

        
def _fasta_itr_from_web(file_handle) :
    "Iterate through a fasta file posted on the web."

    h = file_handle.readline().decode("utf-8")[:-1]
    if h[0] != '>':
        raise MalformedInput()
    h = h[1:]

    seq = []
    for line in file_handle:
        line = line.decode("utf-8")[:-1] # remove newline
        if line[0] == '>':
            yield FastaRecord(h,''.join(seq))
            h = line[1:]
            seq = []
            continue
        seq.append(line)

    yield FastaRecord(h,''.join(seq))



def _fasta_itr_from_name(fname):
    "Iterate through a fasta file with the given name."

    f = myopen(fname)
    for rec in _fasta_itr_from_file(f) :
        yield rec


def _fasta_itr(src):
    """Provide an iteration through the fasta records in file `src'.
    
    Here `src' can be either a file name or a url of a file.
    """
    if type(src) == str :
        if src.find("http")>=0 :
            file_handle = urlopen(src)
            return _fasta_itr_from_web(file_handle)
        else :
            return _fasta_itr_from_name(src)
    else:
        raise TypeError

    
class fasta_itr (object) :
    """An iterator through a Fasta file"""

    def __init__(self, src) :
        """Create an iterator through the records in src."""
        self.__itr = _fasta_itr(src)

    def __iter__(self) :
        return self

    def __next__(self) :
        return self.__itr.__next__()


In [12]:
data = np.load("dataset.npz")
Y = data["Y"]
windows = data["windows"]

genome = {}
genome_files = ["data/fasta/Arabidopsis_thaliana.TAIR10.dna.chromosome.1.fa", "data/fasta/Arabidopsis_thaliana.TAIR10.dna.chromosome.2.fa", "data/fasta/Arabidopsis_thaliana.TAIR10.dna.chromosome.3.fa",
                "data/fasta/Arabidopsis_thaliana.TAIR10.dna.chromosome.4.fa", "data/fasta/Arabidopsis_thaliana.TAIR10.dna.chromosome.5.fa"]

for loc in genome_files:
    for record in fasta_itr(loc):
        chr = "Chr" + record.header.split()[0] # building chr name from number in the arapidopsis file
        genome[chr] = record.sequence.upper()

onehot = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

X = np.zeros((len(windows), 4, winsize))

for i, (chr, start, end) in enumerate(windows):
    seq = genome[chr][int(start):int(end)]
    for j, base in enumerate(seq):
        if base in onehot:
            X[i, onehot[base], j] = 1.0

print(X.shape)

np.savez_compressed("fulldataset.npz", X=X, Y=Y, windows=windows)

(59570, 4, 2000)


In [2]:
# how to load dataset from our files, i know saving Y again is redundant but whatever
import numpy as np

fulldata = np.load("fulldataset.npz")
X = fulldata["X"]
Y = fulldata["Y"]
windows = fulldata["windows"]

### **Architectures** ###

#### Basset (Multi-layer CNN) ####

#### Transformer ####

### **Analysis of Results** ###

## Enhancer Activity in Arabidopsis ##

#### Data Pre-Processing ####
Done by Ben Carter

In [6]:
AMBIGUOUS_BASES = {
    'N': ['A', 'C', 'G', 'T'],
    'W': ['A', 'T'],
    'S': ['C', 'G'],
    'Y': ['C', 'T'],
    'K': ['G', 'T'],
    'M': ['A', 'C'],
    'R': ['A', 'G'],
}

def clean_seq(sequences):
    cleaned = []
    for seq in sequences:
        if len(seq) == 2500:
            cleaned.append(''.join(
                random.choice(AMBIGUOUS_BASES[base]) if base in AMBIGUOUS_BASES else base
                for base in seq
            ))
    return np.array(cleaned)

def one_hot_encode(sequences):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T':[0,0,0,1]}
    encoded = np.array([[mapping[base] for base in seq] for seq in sequences])
    return encoded.swapaxes(1,2)

In [9]:
filename='data/arabidopsis_enhancers.csv'
chrs = ["Chr1", "Chr2", "Chr3", "Chr4", "Chr5"]
data = pd.read_csv(filename,sep=',',header=0).values
train_data = data[data[:,4] == "Train"]
test_data = data[data[:,4] == "Test"]

train_seqs = train_data[:,3]
train_labels = train_data[:,5]

test_seqs = test_data[:,3]
test_labels = test_data[:,5]

train_X = torch.from_numpy(one_hot_encode(clean_seq(train_seqs)))
test_X = torch.from_numpy(one_hot_encode(clean_seq(test_seqs)))


In [ ]:
print(train_X.shape)
print(test_X.shape)

torch.Size([45708, 4, 2500])


Use of AI and web source in this section:
* Claude Code used to help optimize clean function due to it taking a long time to execute.

### **Architectures** ###

#### Multi-layer CNN without Transfer Learning ####

#### Multi-layer CNN with Transfer Learning ####

#### Fine-tuned Transformer ####

### **Analysis of Results** ###

# **Conclusion** #